# 🚀 Autonomous Code Review Squad - Local SLM Fine-Tuning Pipeline
This Google Colab notebook automates the QLoRA 4-bit fine-tuning of 3 specialized Small Language Models (SLMs) using **Unsloth**:
1. **Security Reviewer (`security-reviewer-1.5b`)** - OWASP Top 10, CWE-89 SQL Injection & Hardcoded Secret Detector
2. **Clean Code Reviewer (`clean-code-reviewer-3b`)** - SOLID, Async/Await anti-patterns & Cognitive Complexity
3. **Unit Test Generator (`unittest-generator-1.3b`)** - xUnit & Moq Test Suite Generator

Models are fine-tuned with **ChatML schema enforcement**, verified in-memory for raw JSON responses (no Markdown wrappers), exported to **GGUF `q4_k_m`**, and automatically downloaded for Ollama Docker deployment.

## Section 1: Environment & GPU Setup

In [ ]:
# Install Unsloth, unsloth_zoo, and fine-tuning dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install --no-deps xformers trl peft accelerate bitsandbytes

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected"
print(f"✅ Active GPU Instance: {gpu_name}")
assert torch.cuda.is_available(), "❌ GPU instance is required! Enable GPU acceleration in Runtime -> Change runtime type."

## Section 2: Configuration & Model Selector (Parameterized)

In [ ]:
# @title 🎯 Select Agent Target to Fine-Tune
TARGET_AGENT = "security" # @param ["security", "cleancode", "unittest"]

AGENT_CONFIGS = {
    "security": {
        "model_name": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
        "export_name": "security-reviewer-1.5b",
        "system_prompt": "You are an Enterprise Security Reviewer. You MUST return raw JSON without Markdown wrappers matching schema: {\"has_vulnerability\": boolean, \"vulnerabilities\": [{\"cwe_id\": string, \"severity\": string, \"line_number\": int, \"title\": string, \"description\": string, \"remediation_code\": string}]}"
    },
    "cleancode": {
        "model_name": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "export_name": "clean-code-reviewer-3b",
        "system_prompt": "You are a Clean Code Reviewer. You MUST return raw JSON without Markdown wrappers matching schema: {\"refactoring_suggestions\": [{\"category\": string, \"principle\": string, \"line_number\": int, \"issue\": string, \"suggested_code\": string}]}"
    },
    "unittest": {
        "model_name": "unsloth/DeepSeek-Coder-1.3B-Instruct-bnb-4bit",
        "export_name": "unittest-generator-1.3b",
        "system_prompt": "You are a Unit Test Generator. You MUST return raw JSON without Markdown wrappers matching schema: {\"test_class_name\": string, \"target_framework\": \"xUnit_Moq\", \"mocked_interfaces\": [string], \"complete_test_code\": string}"
    }
}

selected_config = AGENT_CONFIGS[TARGET_AGENT]
print(f"🎯 Training Selected Agent: {TARGET_AGENT.upper()}")
print(f"📦 Base Model: {selected_config['model_name']}")
print(f"💾 Output GGUF Target: {selected_config['export_name']}.gguf")

## Section 3: Embedded Seed Dataset (The 7 Golden Standard Records)

In [ ]:
import json
import os

# 7 Golden Standard ChatML dataset records matching backend ACL & Roslyn AST schemas
SEED_DATASETS = {
    "security": [
        {
            "messages": [
                {"role": "system", "content": selected_config["system_prompt"]},
                {"role": "user", "content": json.dumps({"file_path": "Services/UserService.cs", "class_name": "UserService", "injected_dependencies": ["DbContext"], "target_method": {"signature": "async Task<object> GetUserByEmail(string email)", "body": "var sql = \"SELECT * FROM Users WHERE Email = '\" + email + \"'\"; return await _context.Set<object>().FromSqlRaw(sql).FirstOrDefaultAsync();"}})},
                {"role": "assistant", "content": json.dumps({"has_vulnerability": True, "vulnerabilities": [{"cwe_id": "CWE-89", "severity": "Critical", "line_number": 14, "title": "SQL Injection Vulnerability", "description": "Raw string concatenation in FromSqlRaw allows arbitrary SQL execution.", "remediation_code": "var user = await _context.Users.FromSqlInterpolated($\"SELECT * FROM Users WHERE Email = {email}\").FirstOrDefaultAsync();"}]})}
            ]
        },
        {
            "messages": [
                {"role": "system", "content": selected_config["system_prompt"]},
                {"role": "user", "content": json.dumps({"file_path": "Services/AuthService.cs", "class_name": "AuthService", "injected_dependencies": ["IConfiguration"], "target_method": {"signature": "string GetJwtSecret()", "body": "private const string Secret = \"sk_live_99481948104810294\"; return Secret;"}})},
                {"role": "assistant", "content": json.dumps({"has_vulnerability": True, "vulnerabilities": [{"cwe_id": "CWE-798", "severity": "High", "line_number": 8, "title": "Hardcoded API Secret", "description": "Private API key is hardcoded directly in source code.", "remediation_code": "return _configuration[\"Jwt:Secret\"] ?? throw new InvalidOperationException();"}]})}
            ]
        }
    ],
    "cleancode": [
        {
            "messages": [
                {"role": "system", "content": selected_config["system_prompt"]},
                {"role": "user", "content": json.dumps({"file_path": "Services/OrderService.cs", "class_name": "OrderService", "injected_dependencies": ["IOrderRepository"], "target_method": {"signature": "object GetOrderSync(int id)", "body": "return GetOrderByIdAsync(id).Result;"}})},
                {"role": "assistant", "content": json.dumps({"refactoring_suggestions": [{"category": "Async/Await Anti-Pattern", "principle": "Non-blocking I/O", "line_number": 22, "issue": "Sync-over-async via .Result causes thread pool starvation and deadlock risk.", "suggested_code": "return await GetOrderByIdAsync(id, cancellationToken);"}]})}
            ]
        }
    ],
    "unittest": [
        {
            "messages": [
                {"role": "system", "content": selected_config["system_prompt"]},
                {"role": "user", "content": json.dumps({"file_path": "Services/PaymentProcessor.cs", "class_name": "PaymentProcessor", "injected_dependencies": ["IPaymentGateway"], "target_method": {"signature": "async Task ProcessPaymentAsync(PaymentRequest request)", "body": "if (request == null) throw new ArgumentNullException(nameof(request)); return _gateway.ChargeAsync(request);"}})},
                {"role": "assistant", "content": json.dumps({"test_class_name": "PaymentProcessorTests", "target_framework": "xUnit_Moq", "mocked_interfaces": ["IPaymentGateway"], "complete_test_code": "[Fact]\npublic async Task ProcessPaymentAsync_NullRequest_ThrowsArgumentNullException()\n{\n    var gatewayMock = new Mock<IPaymentGateway>();\n    var processor = new PaymentProcessor(gatewayMock.Object);\n    await Assert.ThrowsAsync<ArgumentNullException>(() => processor.ProcessPaymentAsync(null!));\n}"})}
            ]
        }
    ]
}

dataset_filename = "train_dataset.jsonl"
active_records = SEED_DATASETS.get(TARGET_AGENT, SEED_DATASETS["security"])
with open(dataset_filename, "w", encoding="utf-8") as f:
    for record in active_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"✅ Prepared {len(active_records)} training records in '{dataset_filename}'.")

## Section 4: QLoRA Model & Adapter Initialization

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = selected_config["model_name"],
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype = None
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407
)
print(f"✅ Initialized QLoRA 4-bit PEFT Model for {TARGET_AGENT.upper()}")

## Section 5: Data Formatting & train_on_responses_only

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_sharegpt
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

def format_prompts(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

raw_dataset = load_dataset("json", data_files=dataset_filename, split="train")
dataset = raw_dataset.map(format_prompts, batched=True)
print("✅ Dataset formatted with ChatML template.")

## Section 6: SFTTrainer Execution

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Mask user inputs to compute loss EXCLUSIVELY on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer.train()
print(f"🎉 Fine-tuning finished for {TARGET_AGENT.upper()}!")

## Section 7: In-Memory Inference Verification

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = {
    "file_path": "Services/RefundService.cs",
    "class_name": "RefundService",
    "injected_dependencies": ["DbContext"],
    "target_method": {
        "signature": "async Task ProcessRefund(string txId, decimal amount)",
        "body": "var sql = \"SELECT * FROM Tx WHERE Id = '\" + txId + \"'\"; return await _context.FromSqlRaw(sql).ToListAsync();"
    }
}

messages = [
    {"role": "system", "content": selected_config["system_prompt"]},
    {"role": "user", "content": json.dumps(test_prompt)}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1)
decoded = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("🔍 IN-MEMORY INFERENCE OUTPUT:")
print(decoded)
print("\n✅ Output Verification: Confirm that response is a RAW valid JSON string without ```json wrappers.")

## Section 8: GGUF (q4_k_m) Export & Auto-Download

In [ ]:
# Ensure unsloth_zoo is installed for Llama 3.2 GGUF conversion
!pip install -q unsloth_zoo

export_filename = selected_config["export_name"]
model.save_pretrained_gguf(export_filename, tokenizer, quantization_method = "q4_k_m")

gguf_filepath = f"{export_filename}-unsloth.Q4_K_M.gguf"
print(f"✅ Successfully exported model to GGUF format: {gguf_filepath}")

try:
    from google.colab import files
    files.download(gguf_filepath)
    print(f"📥 Triggered auto-download of '{gguf_filepath}' to your local machine.")
except Exception as e:
    print(f"ℹ️ Download manually from Colab sidebar files: {gguf_filepath}")

## Section 9: Infrastructure Integration Instructions

### How to Deploy Fine-Tuned SLMs to Docker Compose Ollama Infrastructure:

1. **Move Downloaded GGUF Files:**
   Place the downloaded `.gguf` file into the `./models/` folder in your project repository:
   - `./models/security-reviewer-1.5b.gguf`
   - `./models/clean-code-reviewer-3b.gguf`
   - `./models/unittest-generator-1.3b.gguf`

2. **Create Ollama Modelfile (`./models/Modelfile.security`):**
   ```dockerfile
   FROM ./security-reviewer-1.5b.gguf
   PARAMETER temperature 0.1
   SYSTEM "You are an Enterprise Security Reviewer. Return raw JSON matching contract."
   ```

3. **Register Models with Ollama:**
   ```bash
   docker exec -it smart-review-ollama ollama create security-reviewer -f /root/.ollama/Modelfile.security
   docker exec -it smart-review-ollama ollama create clean-code-reviewer -f /root/.ollama/Modelfile.cleancode
   docker exec -it smart-review-ollama ollama create unittest-generator -f /root/.ollama/Modelfile.unittest
   ```

4. **Enable Custom SLMs in Backend:**
   Set `"Ollama:UseOfflineMockFallback": false` in `appsettings.json` to switch from offline mock fallback to live fine-tuned SLM execution!